# Week 2 workshop · Build, measure, explain

<div class="reader-route">
  <div class="reader-route-label">Route through the workshop</div>
  <div class="reader-route-body">Generate → test → visualise → rasterise → count boxes → estimate dimension → interrogate bias</div>
</div>

## Learning outcomes

By the end of the workshop, you should be able to:

- implement recursive constructions without hidden global state;
- test the scaling properties of the Cantor set and Sierpiński triangle;
- generate a reproducible fractal using an iterated function system;
- estimate a box-counting dimension efficiently;
- explain why a finite raster estimate differs from an exact fractal dimension;
- distinguish mathematical structure from plotting and measurement artefacts.

Predict first. Test a small case. Then increase the depth or resolution.


In [ ]:
from typing import Sequence

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection

SEED = 3024
INK = "#1B2A4C"
YELLOW = "#EDCC55"
BLUE = "#5879AA"

rng = np.random.default_rng(SEED)
print(f"NumPy {np.__version__} · seed {SEED}")


# Construction I · The Cantor set

At each step, replace every interval by its left and right thirds. Depth zero is the original interval.

<div class="discussion-marker"><img src="images/discussion_marker.svg" alt="Discussion prompt"><span>At depth $d$, how many intervals remain, how long is each interval, and what is their total length?</span></div>


In [ ]:
def cantor_intervals(
    depth: int,
    start: float = 0.0,
    end: float = 1.0,
) -> np.ndarray:
    """Return the depth-`depth` Cantor intervals as an array of shape (2**depth, 2)."""
    if not isinstance(depth, (int, np.integer)) or depth < 0:
        raise ValueError("depth must be a non-negative integer")
    if not start < end:
        raise ValueError("start must be smaller than end")
    if depth == 0:
        return np.array([[start, end]], dtype=float)

    third = (end - start) / 3.0
    left = cantor_intervals(depth - 1, start, start + third)
    right = cantor_intervals(depth - 1, end - third, end)
    return np.vstack((left, right))


## Turn the definition into tests

An executable test records what the mathematics says must be true. It also makes later refactoring safer.


In [ ]:
for depth in range(7):
    intervals = cantor_intervals(depth)
    lengths = intervals[:, 1] - intervals[:, 0]
    assert len(intervals) == 2**depth
    assert np.allclose(lengths, 3.0**(-depth))
    assert np.isclose(lengths.sum(), (2.0 / 3.0) ** depth)

assert np.allclose(cantor_intervals(1), [[0, 1/3], [2/3, 1]])
print("All Cantor-set tests passed.")


In [ ]:
def plot_cantor_iterations(max_depth: int = 6):
    """Plot construction depths without redefining the Cantor-set function."""
    fig, ax = plt.subplots(figsize=(10, 4))
    for depth in range(max_depth + 1):
        intervals = cantor_intervals(depth)
        segments = [[(left, depth), (right, depth)] for left, right in intervals]
        ax.add_collection(LineCollection(segments, colors=INK, linewidths=2))
    ax.set(xlim=(0, 1), ylim=(-0.5, max_depth + 0.5), xlabel="Position", ylabel="Depth")
    ax.invert_yaxis()
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.set_yticks(range(max_depth + 1))
    return fig, ax


plot_cantor_iterations()
plt.show()


<div class="ladder-marker"><img src="images/ladder_marker.svg" alt="Ladder of abstraction"><span><strong>Up one rung:</strong> the construction gives a scaling law: two copies, each reduced by a factor of three.</span></div>

The similarity dimension is therefore

$$D=\frac{\log 2}{\log 3}\approx0.631.$$


# Construction II · The Sierpiński triangle

At each step, replace one triangle by three half-scale copies.

<div class="discussion-marker"><img src="images/discussion_marker.svg" alt="Discussion prompt"><span>Predict the number of triangles and the remaining area fraction at depth $d$.</span></div>


In [ ]:
EQUILATERAL = np.array(
    [[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3.0) / 2.0]]
)


def sierpinski_triangles(depth: int, triangle: np.ndarray = EQUILATERAL) -> np.ndarray:
    """Return the filled triangles remaining after `depth` recursive subdivisions."""
    if not isinstance(depth, (int, np.integer)) or depth < 0:
        raise ValueError("depth must be a non-negative integer")
    triangle = np.asarray(triangle, dtype=float)
    if triangle.shape != (3, 2):
        raise ValueError("triangle must contain three 2D vertices")
    if depth == 0:
        return triangle[np.newaxis, :, :]

    a, b, c = triangle
    ab, bc, ca = (a + b) / 2, (b + c) / 2, (c + a) / 2
    children = (
        np.array([a, ab, ca]),
        np.array([ab, b, bc]),
        np.array([ca, bc, c]),
    )
    return np.concatenate([sierpinski_triangles(depth - 1, child) for child in children])


In [ ]:
def triangle_area(triangle: np.ndarray) -> float:
    a, b, c = triangle
    twice_area = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
    return abs(twice_area) / 2


original_area = triangle_area(EQUILATERAL)
for depth in range(6):
    triangles = sierpinski_triangles(depth)
    assert len(triangles) == 3**depth
    remaining_area = sum(triangle_area(t) for t in triangles)
    assert np.isclose(remaining_area / original_area, (3.0 / 4.0) ** depth)

print("All Sierpiński-triangle tests passed.")


In [ ]:
def plot_sierpinski_iterations(depths: Sequence[int] = range(6)):
    fig, axes = plt.subplots(1, len(depths), figsize=(12, 2.4))
    axes = np.atleast_1d(axes)
    for ax, depth in zip(axes, depths):
        collection = PolyCollection(
            sierpinski_triangles(depth), facecolor=INK, edgecolor="none"
        )
        ax.add_collection(collection)
        ax.set(xlim=(0, 1), ylim=(0, np.sqrt(3)/2), aspect="equal", title=f"Depth {depth}")
        ax.axis("off")
    fig.tight_layout()
    return fig, axes


plot_sierpinski_iterations()
plt.show()


The exact similarity dimension follows directly from three copies scaled by one half:

$$D_{\mathrm{exact}}=\frac{\log 3}{\log 2}\approx1.585.$$

We will now pretend we do not know the construction and estimate this dimension from finite data.


# Generate observations with an IFS

The chaos game repeatedly chooses a vertex and moves halfway towards it. Randomness chooses the sequence, while the three contraction maps define the attractor.

<div class="choice-marker"><img src="images/choice_marker.svg" alt="Modelling choice"><span>The seed, number of points, burn-in, and starting point are experimental choices.</span></div>


In [ ]:
def sierpinski_chaos_game(
    n_points: int,
    rng: np.random.Generator,
    burn_in: int = 20,
) -> np.ndarray:
    """Generate reproducible points on the Sierpiński attractor."""
    if n_points < 1 or burn_in < 0:
        raise ValueError("n_points must be positive and burn_in non-negative")
    point = EQUILATERAL.mean(axis=0)
    points = np.empty((n_points, 2), dtype=float)
    for step in range(n_points + burn_in):
        vertex = EQUILATERAL[rng.integers(0, 3)]
        point = 0.5 * (point + vertex)
        if step >= burn_in:
            points[step - burn_in] = point
    return points


points = sierpinski_chaos_game(150_000, np.random.default_rng(SEED))
assert points.shape == (150_000, 2)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(points[:, 0], points[:, 1], s=0.08, color=INK, rasterized=True)
ax.set(aspect="equal", title="Reproducible chaos-game sample")
ax.axis("off")
plt.show()


# Rasterise without writing files

The previous workshop saved and reloaded figures. That mixed the mathematical object with axes, margins, antialiasing, and file formats. Here we rasterise the point coordinates directly into a Boolean array.


In [ ]:
def rasterise_points(points: np.ndarray, resolution: int = 512) -> np.ndarray:
    """Map 2D points directly to an occupied-pixel array."""
    points = np.asarray(points, dtype=float)
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("points must have shape (n, 2)")
    if resolution < 2:
        raise ValueError("resolution must be at least two")
    minimum, maximum = points.min(axis=0), points.max(axis=0)
    span = maximum - minimum
    if np.any(span == 0):
        raise ValueError("points must span both coordinate directions")
    scaled = (points - minimum) / span
    indices = np.clip((scaled * (resolution - 1)).astype(int), 0, resolution - 1)
    image = np.zeros((resolution, resolution), dtype=bool)
    image[indices[:, 1], indices[:, 0]] = True
    return image


fractal_image = rasterise_points(points, resolution=512)
print(f"occupied pixels: {fractal_image.sum():,} of {fractal_image.size:,}")
plt.figure(figsize=(5, 5))
plt.imshow(fractal_image, cmap="Greys", origin="lower")
plt.axis("off")
plt.show()


# Count occupied boxes efficiently

For box width $\varepsilon$, let $N(\varepsilon)$ be the number of boxes containing at least one occupied pixel. If

$$N(\varepsilon)\propto\varepsilon^{-D},$$

then the slope of $\log N$ against $\log(1/\varepsilon)$ estimates $D$.

The implementation reshapes the array into blocks and reduces each block once. It does not create thousands of plotting rectangles.


In [ ]:
def occupied_blocks(image: np.ndarray, box_size: int) -> np.ndarray:
    """Return a Boolean grid indicating which non-overlapping boxes are occupied."""
    image = np.asarray(image, dtype=bool)
    if image.ndim != 2:
        raise ValueError("image must be two-dimensional")
    height, width = image.shape
    if box_size < 1 or height % box_size or width % box_size:
        raise ValueError("box_size must be a positive divisor of both image dimensions")
    return image.reshape(
        height // box_size, box_size, width // box_size, box_size
    ).any(axis=(1, 3))


def box_counts(image: np.ndarray, box_sizes: Sequence[int]) -> np.ndarray:
    """Count occupied boxes at each requested scale."""
    sizes = np.asarray(box_sizes, dtype=int)
    if sizes.ndim != 1 or sizes.size < 2:
        raise ValueError("provide at least two box sizes")
    return np.array([occupied_blocks(image, int(size)).sum() for size in sizes])


test_image = np.eye(8, dtype=bool)
assert np.array_equal(box_counts(test_image, [1, 2, 4]), [8, 4, 2])
print("Box-counting test passed.")


In [ ]:
box_sizes = np.array([2, 4, 8, 16, 32, 64])
counts = box_counts(fractal_image, box_sizes)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, size in zip(axes, [8, 32, 64]):
    blocks = occupied_blocks(fractal_image, size)
    ax.imshow(blocks, cmap="Greys", interpolation="nearest", origin="lower")
    ax.set(title=f"Box width {size}\n{blocks.sum()} occupied", xticks=[], yticks=[])
fig.tight_layout()
plt.show()


In [ ]:
def fit_box_dimension(
    image: np.ndarray,
    box_sizes: Sequence[int],
) -> tuple[float, float, np.ndarray]:
    """Return slope, intercept, and counts for a specified finite scale range."""
    sizes = np.asarray(box_sizes, dtype=int)
    counts = box_counts(image, sizes)
    if np.any(counts <= 0):
        raise ValueError("all selected scales must contain occupied boxes")
    inverse_scale = image.shape[0] / sizes
    slope, intercept = np.polyfit(np.log(inverse_scale), np.log(counts), 1)
    return float(slope), float(intercept), counts


estimate, intercept, counts = fit_box_dimension(fractal_image, box_sizes)
exact = np.log(3) / np.log(2)
x = np.log(fractal_image.shape[0] / box_sizes)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, np.log(counts), "o", color=BLUE, label="Measured counts")
ax.plot(x, estimate * x + intercept, color=INK, label=f"Fit: D = {estimate:.3f}")
ax.set(xlabel=r"$\log(1/\varepsilon)$", ylabel=r"$\log N(\varepsilon)$")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()

print(f"finite raster estimate: {estimate:.4f}")
print(f"exact similarity dimension: {exact:.4f}")
print(f"absolute difference: {abs(estimate-exact):.4f}")
assert abs(estimate - exact) < 0.1


# Interrogate the estimate

The estimate should not equal the exact dimension perfectly. We have replaced an infinite set and a limit as $\varepsilon\to0$ with:

- finitely many sampled points;
- a finite pixel grid;
- square boxes aligned to one origin;
- a short and subjective scale range;
- an ordinary least-squares line fit.

<div class="discussion-marker"><img src="images/discussion_marker.svg" alt="Discussion prompt"><span>Which of these choices could move the estimate up or down? Which can be checked without knowing the exact answer?</span></div>


In [ ]:
scale_ranges = {
    "fine to medium": [2, 4, 8, 16, 32],
    "middle": [4, 8, 16, 32],
    "medium to coarse": [8, 16, 32, 64],
}

for label, sizes in scale_ranges.items():
    dimension, _, _ = fit_box_dimension(fractal_image, sizes)
    print(f"{label:>16}: D = {dimension:.4f}")


<div class="ladder-marker"><img src="images/ladder_marker.svg" alt="Ladder of abstraction"><span><strong>Down:</strong> inspect pixels and occupied boxes. <strong>Up:</strong> compress their scaling relationship into one estimated exponent.</span></div>

A dimension estimate without its resolution, scale range, and uncertainty is incomplete evidence.


# Choose an extension

### A · Sampling robustness

Repeat the chaos game with several independent seeds and sample sizes. Report the distribution of estimated dimensions.

### B · Grid sensitivity

Shift the box-grid origin or change raster resolution. Identify a scale range that is reasonably stable.

### C · Another construction

Implement the Sierpiński carpet. Compare its exact similarity dimension with a finite raster estimate.

### D · Natural data

Apply the method to a supplied coastline or branching image. Document thresholding, cropping, resolution, and the scale range used.


In [ ]:
# Your extension goes here. Keep the baseline functions unchanged where possible.


# Exit ticket

In three sentences:

1. State the scaling relationship used to define or estimate dimension.
2. Name one computational choice that affected the estimate.
3. Explain why a straight-looking log–log plot is not sufficient evidence on its own.
